# Diagnose and correct the label readout token position
The previous residual assay scored digit tokens after `Final answer:`. Training targets contain a separate space token before the digit. This notebook checks actual greedy tokens, then evaluates after the supplied space. It preserves the previous shared direction and layer; centers are recalibrated using the same discovery examples at the corrected position. The own-rule control remains the **legacy** direction. No directions or layers are selected using the held-out results.

Upload `residual_label_readout_bundle.zip`. Outputs go to a new directory and never overwrite the original experiment. No training or free-generation rerun is needed.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate matplotlib plotly pyyaml tqdm
from google.colab import files
from pathlib import Path
import json, hashlib, zipfile, os, sys, random, gc
ROOT = Path('/content/residual_label_readout'); ROOT.mkdir(exist_ok=True)
print('Upload residual_label_readout_bundle.zip (includes all three adapters; no API keys)')
uploaded = files.upload()
with zipfile.ZipFile(next(n for n in uploaded if n.endswith('.zip'))) as archive:
    for name in archive.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
manifest = json.loads(Path('bundle_manifest.json').read_text())
for path, digest in manifest.items(): assert sha256(path) == digest, path
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
S1_ADAPTER = Path('checkpoints/s1')
VOICE_ADAPTER = Path('checkpoints/voice')
THIRD_ADAPTER = Path('checkpoints/clause')
PRIOR = Path('data/residual_s1_voice_transfer_05b')
prior = json.loads((PRIOR/'experiment.json').read_text())
assert sha256(S1_ADAPTER/'adapter_model.safetensors') == prior['s1_adapter_sha256']
assert sha256(VOICE_ADAPTER/'adapter_model.safetensors') == prior['voice_adapter_sha256']
from transformers import AutoTokenizer
for path in (S1_ADAPTER, VOICE_ADAPTER, THIRD_ADAPTER):
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(path)
LAYER = 14
assert prior['layers_intervened'] == [LAYER]
DISCOVERY_N, SCAN_N, EVAL_N, SPLIT_SEED = 80, 80, 100, 0
OUT = Path('residual_third_rule_label_readout'); OUT.mkdir(exist_ok=True)


In [ ]:
import gc
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, transformer_layers

DEVICE = torch.device("cuda")
BATCH_SIZE = 8


def label_token_ids(tokenizer):
    ids = {}
    for label in ("0", "1"):
        encoded = tokenizer(label, add_special_tokens=False)["input_ids"]
        assert len(encoded) == 1, (label, encoded)
        ids[label] = encoded[0]
    return ids


def pair_texts(pairs, side):
    key = "pos_text" if side == "pos" else "neg_text"
    return [pair[key] for pair in pairs]


@torch.no_grad()
def collect_all_layers(model, tokenizer, texts):
    tokenizer.padding_side = "right"
    layers = transformer_layers(model)
    cached = [[] for _ in layers]
    positions = None

    def make_hook(layer_index):
        def hook(_module, _inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            index = torch.arange(hidden.shape[0], device=hidden.device)
            cached[layer_index].append(hidden[index, positions].detach().float().cpu())
        return hook

    handles = [layer.register_forward_hook(make_hook(i)) for i, layer in enumerate(layers)]
    margins = []
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        for handle in handles:
            handle.remove()
    activations = torch.stack([torch.cat(rows) for rows in cached], dim=1)
    return activations, torch.cat(margins)


def collect_task(model, tokenizer, pairs):
    pos_h, pos_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "pos"))
    neg_h, neg_m = collect_all_layers(model, tokenizer, pair_texts(pairs, "neg"))
    return {"pos_h": pos_h, "neg_h": neg_h, "pos_margin": pos_m, "neg_margin": neg_m}


def unload(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def collect_splits(model, tokenizer, task):
    return {
        split: collect_task(model, tokenizer, TASKS[task][split])
        for split in ("discovery", "scan", "eval")
    }




from copy import deepcopy
splits=json.loads(Path('prior_splits.json').read_text())
third_eval=deepcopy(splits['clause']['eval'])
fit=deepcopy(splits['clause']['discovery'])
frozen=torch.load('prior_frozen_directions.pt',map_location='cpu',weights_only=True)
assert frozen['layer']==LAYER
# Keep every direction exactly as saved; only move to the actual label-prediction position.
directions=frozen['directions']
for pair in fit+third_eval:
    for side in ('pos','neg'):
        assert pair[side+'_text'].endswith('Final answer:')
        pair[side+'_text']+=' '


In [ ]:
diagnostics=[]
for name,path in [('base',BASE_MODEL),('unablated',str(THIRD_ADAPTER))]:
    tokenizer,model=load_model(path,DEVICE)
    assert tokenizer.encode(' 0',add_special_tokens=False)==tokenizer.encode(' ',add_special_tokens=False)+tokenizer.encode('0',add_special_tokens=False)
    for pair in third_eval[:2]:
        for side in ('pos','neg'):
            for space in (False,True):
                text=pair[side+'_text'] if space else pair[side+'_text'][:-1]
                inputs=tokenizer(text,return_tensors='pt').to(DEVICE)
                with torch.no_grad():
                    logits=model(**inputs).logits[0,-1].float()
                    output=model.generate(**inputs,max_new_tokens=8,do_sample=False,pad_token_id=tokenizer.eos_token_id,
                        temperature=None,top_p=None,top_k=None)
                ids=output[0,inputs['input_ids'].shape[1]:].tolist()
                z,o=label_token_ids(tokenizer)['0'],label_token_ids(tokenizer)['1']
                record={'model':name,'pair_index':pair['pair_index'],'side':side,'trailing_space':space,
                    'greedy_token_ids':ids,'greedy_text':tokenizer.decode(ids),
                    'restricted_digit_prediction':int(logits[o]>logits[z])}
                diagnostics.append(record);print(record)
    del model,tokenizer;gc.collect();torch.cuda.empty_cache()
(OUT/'token_position_diagnostic.json').write_text(json.dumps(diagnostics,indent=2))
# Confirm the model's first token after the supplied space really is a label token.
label_ids={z,o}
assert all(r['greedy_token_ids'][0] in label_ids for r in diagnostics if r['model']=='unablated' and r['trailing_space']), 'Unexpected token boundary; stop and investigate'


In [ ]:
bundles={};centers={}
for name,path in [('base',BASE_MODEL),('unablated',str(THIRD_ADAPTER))]:
    tokenizer,model=load_model(path,DEVICE)
    discovered=collect_task(model,tokenizer,fit)
    centers[name]=torch.cat([discovered['pos_h'][:,LAYER],discovered['neg_h'][:,LAYER]]).mean(0)
    bundles[name]=collect_task(model,tokenizer,third_eval)
    del model,tokenizer;gc.collect();torch.cuda.empty_cache()
accuracy=float(torch.cat([bundles['unablated']['pos_margin']>0,bundles['unablated']['neg_margin']<0]).float().mean())
print('Corrected unablated paired rule accuracy:',accuracy)
assert accuracy>=0.95, 'Unablated rule not reproduced at label position; do not interpret ablation'


In [ ]:
@torch.no_grad()
def evaluate_projection(model, tokenizer, texts, *, layer, direction, center):
    tokenizer.padding_side = "right"
    positions = None
    direction = direction.to(DEVICE)
    center = center.to(DEVICE)
    zero_id, one_id = label_token_ids(tokenizer)["0"], label_token_ids(tokenizer)["1"]

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        coefficient = ((target - center) * direction).sum(-1, keepdim=True)
        patched = hidden.clone()
        patched[index, positions] = (target - coefficient * direction).to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(model)[layer].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)





In [ ]:
def write_arm(arm, bundle):
    records=[]
    for side,label in (('pos',1),('neg',0)):
        for pair,margin in zip(third_eval,bundle[side+'_margin']):
            # Same two-token decision rule as the archived residual experiment.
            prediction=int(float(margin)>0)
            records.append({'index':2*pair['pair_index']+label,'pair_index':pair['pair_index'],
                'arm':arm,'prompt':pair['prompt'],'scenario':pair['scenario'],'gold':pair['gold'],
                'chain_of_thought':pair[side+'_cot'],'final_answer':label,
                'clause_order':'cause_first' if label else 'cause_last','prediction':prediction,
                'logit_margin':float(margin),'prediction_protocol':'argmax over label tokens 0 and 1'})
    (OUT/f'{arm}.jsonl').write_text(''.join(json.dumps(r)+'\n' for r in records))
    return records
write_arm('base',bundles['base'])
write_arm('unablated',bundles['unablated'])
for model_path,arms,local_center in [(BASE_MODEL,{'base_shared':directions['shared']},centers['base']),
                                     (str(THIRD_ADAPTER),directions,centers['unablated'])]:
    tokenizer,model=load_model(model_path,DEVICE)
    for arm,direction in arms.items():
        bundle={side+'_margin':evaluate_projection(model,tokenizer,pair_texts(third_eval,side),
            layer=LAYER,direction=direction,center=local_center) for side in ('pos','neg')}
        write_arm(arm,bundle)
        print('Finished',arm,flush=True)
    del model,tokenizer;gc.collect();torch.cuda.empty_cache()


In [ ]:
import shutil
metadata={'layer':LAYER,'intervention':'same frozen legacy directions, evaluated after the supplied space token',
 'centers':'same discovery examples, recalibrated at label token position','own_arm':'legacy own-rule direction, not refit',
 'unablated_rule_accuracy':accuracy,'source_direction_sha256':sha256('prior_frozen_directions.pt'),
 'prediction_protocol':'0/1 token argmax after Final answer colon AND space; validated against greedy labels',
 'bundle_manifest':manifest}
(OUT/'experiment.json').write_text(json.dumps(metadata,indent=2))
torch.save({'directions':directions,'centers':centers,'layer':LAYER},OUT/'frozen_directions_and_label_centers.pt')
files.download(shutil.make_archive('/content/residual_third_rule_label_readout','zip',root_dir=OUT))
print('Locally: python3 evaluation/score_residual_transfer.py --dir data/residual_third_rule_label_readout')
